In [1]:
import torch
print(torch.__version__)         # PyTorch version
print(torch.version.cuda)        # CUDA version used to build PyTorch
print(torch.cuda.is_available()) # Whether CUDA is available
print(torch.cuda.get_device_name(0))  # (if available)

2.7.1+cu126
12.6
True
NVIDIA RTX A6000


In [ ]:
import os
import numpy as np
from PIL import Image

In [ ]:
def process_images(input_folder, rgb_output_folder, thermal_output_folder):

    # Iterate through all files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith(('.png', '.jpg', '.jpeg')):  # Add or modify extensions as needed
            file_path = os.path.join(input_folder, filename)
            
            # Read the image
            fused_image = np.array(Image.open(file_path))
            
            if fused_image.shape[2] == 4:  # Ensure it's a 4-channel image
                # Separate RGB and thermal channels
                rgb_image = fused_image[:, :, :3]
                thermal_image = fused_image[:, :, 3]
                
                # Save RGB image
                rgb_output_path = os.path.join(rgb_output_folder, f"{filename}")
                Image.fromarray(rgb_image).save(rgb_output_path)
                
                # Save thermal image
                thermal_output_path = os.path.join(thermal_output_folder, f"{filename}")
                Image.fromarray(thermal_image).save(thermal_output_path)
                
                # Copy corresponding label image to label_output_folder
                label_filename = os.path.splitext(filename)[0] + ".png"
                label_src_path = os.path.join("./datasets/ir_seg_dataset/labels", label_filename)
                label_dst_path = os.path.join(label_output_folder, label_filename)
                if os.path.exists(label_src_path):
                    Image.open(label_src_path).save(label_dst_path)
                else:
                    print(f"Label not found for: {filename}")

                print(f"Processed: {filename}")
            else:
                print(f"Skipped: {filename} (not a 4-channel image)")

# Example usage
input_folder = "./datasets/ir_seg_dataset/images"
rgb_output_folder = "./datasets/MFNet/RGB"
thermal_output_folder = "./datasets/MFNet/Thermal"
label_output_folder = "./datasets/MFNet/Label"


In [ ]:
train_file = "./datasets/ir_seg_dataset/train.txt"
val_file = "./datasets/ir_seg_dataset/val.txt"
test_file = "./datasets/ir_seg_dataset/test.txt"
train_val_file = "./datasets/MFNet/train_val.txt"
new_test_file = "./datasets/MFNet/test.txt"
labels_file = "./datasets/ir_seg_dataset/labels.txt"

In [ ]:
import shutil

# Read train and val files, filter out lines containing '_flip', and join them
with open(train_file, 'r') as f:
    train_lines = [line.strip() for line in f if '_flip' not in line]

with open(val_file, 'r') as f:
    val_lines = [line.strip() for line in f if '_flip' not in line]

combined_lines = train_lines + val_lines

with open(train_val_file, 'w') as f:
    for line in combined_lines:
        f.write(f"{line}\n")

with open(test_file, 'r') as src, open(new_test_file, 'w') as dst:
    for line in src:
        dst.write(line)

In [ ]:
process_images(input_folder, rgb_output_folder, thermal_output_folder)